# Leo Video Studio — interactive GPU rendering
Select a GPU runtime. This uses pretrained Wan, not the old shapes model. Start with one economy shot. GPU availability and sufficient memory are not guaranteed. This notebook does not host a public app.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/leo-video-studio.git"
assert "YOUR_USERNAME" not in REPO_URL, "Set your GitHub repository URL"
import subprocess
from pathlib import Path
if not Path("/content/leo-video-studio").exists():
    subprocess.run(["git", "clone", REPO_URL, "/content/leo-video-studio"], check=True)
%cd /content/leo-video-studio
%pip install -r requirements.txt
subprocess.run(["apt-get", "update"], check=True)
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

Persist the project on Drive so finished scenes survive disconnections. Model cache remains temporary unless you set HF_HOME to persistent storage. Mounting Drive gives notebook code access to it.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.environ["STUDIO_DATA"] = "/content/drive/MyDrive/leo-video-studio"
os.environ["HF_HOME"] = "/content/leo-model-cache"
os.environ["CPU_OFFLOAD"] = "1"
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print(torch.cuda.get_device_name(0))

In [ ]:
from planner import make_plan
import jobs
STORY = "A father in a blue shirt embraces his daughter in a yellow coat in a sunlit garden, medium cinematic shot, natural expressions."
plan = make_plan(STORY, style="Realistic", episodes=1, shots=1, characters="Father: short dark hair, blue shirt. Daughter: yellow coat.")
import json
print(json.dumps(plan, indent=2))
# Edit plan here before proceeding.

In [ ]:
PROJECT_ID = jobs.create(plan, "Economy 1.3B / 480p", steps=40, seed=42)
print("Save this project ID:", PROJECT_ID)
# On a later session, replace PROJECT_ID with your saved value instead of creating again.

In [ ]:
for message, outputs in jobs.render(PROJECT_ID):
    print(message, flush=True)

In [ ]:
from IPython.display import Video, display
from google.colab import files
video = jobs.folder(PROJECT_ID) / "episode-01.mp4"
display(Video(str(video), embed=True, width=640))
files.download(str(video))

Export is 1920×1080, upscaled from model resolution. Review quality before making a longer plan. For the browser studio, deploy app.py on a separate GPU host using README.md. This notebook has not been run on Colab in the build workspace.